Importantions of libraries

In [8]:
import matplotlib.pyplot as plt
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
import seaborn as sns
import pandas as pd
import numpy as np
import yfinance as yf

Class Stocks

In [9]:
class Stock:
    def __init__(self, symbol: str, date_init: str, date_end: str = pd.Timestamp.now().date(), interval: str = "1d"):
        self.symbol = symbol
        self.date_init = date_init
        self.date_end = date_end
        self.interval = interval
        self._df: Optional[pd.DataFrame] = None

    @property
    def df(self) -> pd.DataFrame:
        if self._df is None:  # Só baixa se não tiver dados
            self._df = self.create_dataframe()
        return self._df

    def get_symbol(self):
        return self.symbol

    def create_dataframe(self):
        return yf.download(self.symbol, start=self.date_init, end=self.date_end, interval=self.interval).dropna().copy()

    def create_dataframe_close(self):
        return self.df['Close']

    def log_return(self):
        close_df = self.create_dataframe_close()
        return np.log(close_df / close_df.shift(1)).dropna()

    def return_investment(self):
        acumulated_returns = self.create_dataframe_close().pct_change().dropna()
        return (1 + acumulated_returns).cumprod()

    def sharpe_ratio(self):
        rf = 0.01
        if self.interval == "1mo":
            rolling_mean = self.log_return().rolling(window=6).mean() * 12
            rolling_std = self.log_return().rolling(window=6).std() * np.sqrt(12)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio
        elif self.interval == "1wk":
            rolling_mean = self.log_return().rolling(window=60).mean() * 52
            rolling_std = self.log_return().rolling(window=60).std() * np.sqrt(52)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio
        elif self.interval == "1d":
            rolling_mean = self.log_return().rolling(window=252).mean() * 252
            rolling_std = self.log_return().rolling(window=252).std() * np.sqrt(252)
            sharpe_ratio = (rolling_mean - rf) / rolling_std
            return sharpe_ratio
        else:
            return 0

    def sortino_ratio(self):
        returns = self.create_dataframe_close().pct_change().dropna()
        rf = 0.01
        if self.interval == "1mo":
            mar = rf / 12
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=14).std() * np.sqrt(12)
            rolling_mean = excess.rolling(window=14).mean() * 12
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio
        elif self.interval == "1wk":
            mar = rf / 52
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=60).std() * np.sqrt(52)
            rolling_mean = excess.rolling(window=60).mean() * 52
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio
        if self.interval == "1d":
            mar = rf / 252
            excess = returns - mar
            downside = np.minimum(0, excess)
            downside_deviation = downside.rolling(window=252).std() * np.sqrt(252)
            rolling_mean = excess.rolling(window=252).mean() * 252
            sortino_ratio = rolling_mean / downside_deviation
            sortino_ratio = sortino_ratio.replace([np.inf, -np.inf], np.nan)
            return sortino_ratio
        else:
            return 0

# Exemplo de uso
btc_w = Stock("BTC-USD", "2018-01-01", interval="1wk")
spy_w = Stock("SPY", date_init="2018-01-01", interval="1wk")

display(btc_w.df)
display(spy_w.df)

[*********************100%***********************]  1 of 1 completed

Price,Close,High,Low,Open,Volume
Ticker,BTC-USD,BTC-USD,BTC-USD,BTC-USD,BTC-USD
Date,,,,,
2018-01-01,16477.599609,17712.400391,13154.700195,14112.200195,123814400000
2018-01-08,13772.000000,16537.900391,13105.900391,16476.199219,106022199296
2018-01-15,11600.099609,14445.500000,9402.290039,13767.299805,97932879872
2018-01-22,11786.299805,12040.299805,10129.700195,11633.099609,64691999232
2018-01-29,8277.009766,11875.599609,7796.490234,11755.500000,60810019840
...,...,...,...,...,...
2026-03-02,65969.781250,74051.804688,65303.136719,65734.078125,331122778300
2026-03-09,72789.914062,73927.328125,65858.007812,65969.585938,301054621121


[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,SPY,SPY,SPY,SPY,SPY
Date,,,,,
2018-01-01,240.654953,240.778162,235.356338,235.743613,340886500
2018-01-08,244.615692,244.782899,240.214863,240.558114,337325100
2018-01-15,246.807281,246.807281,243.084171,245.874308,461462000
2018-01-22,252.237839,252.281863,246.543166,246.596001,515553600
2018-01-29,242.441696,252.105900,242.406482,251.665817,593556800
...,...,...,...,...,...
2026-03-02,670.548706,686.744465,667.836083,676.851500,478956000
2026-03-09,660.486206,681.498828,659.558746,664.575076,458697100


In [10]:
btc_close = btc_w.create_dataframe_close()

In [11]:
btc_w.sortino_ratio()

Ticker,BTC-USD
Date,
2018-01-08,NaN
2018-01-15,NaN
2018-01-22,NaN
2018-01-29,NaN
2018-02-05,NaN
...,...
2026-03-02,-1.056426
2026-03-09,-0.933302
2026-03-16,-1.226423
